# Notebook 2: Suburb Performance Statistics

## What this notebook covers

The previous notebook downloaded individual property listings, one row per property sold. This notebook downloads something different: **aggregated market statistics** for an entire suburb, summarised by quarter.

Aggregated statistics include things like:
- The median sale price for all houses sold in Carlton in Q3 2023
- How many properties sold in that quarter
- The median number of days a property sat on the market before selling

These come from a different endpoint that works differently from the listings search:

- One call retrieves **up to 45 quarters** of historical data (about 11 years)
- The cost is always **1 credit per call**, regardless of how many quarters you request
- All downstream analysis (charts, comparisons) happens locally from that one fetch

**By the end** you will have quarterly statistics for six Melbourne suburbs in a single table, plus two charts produced from that data without spending any more credits.

---

## Before You Start

**If you have already completed Notebook 0**, your packages, `.env` file, and connection are already set up.

If this is your first time in phase-2, complete `notebook-0-getting-started.ipynb` first. It covers package installation, creating the `.env` credentials file, and making a test call to confirm everything works. Then complete Notebook 1 before this one.

**Important note about request types:** Notebook 1 (the listings endpoint) uses a POST request, where you submit a search form with your criteria. This notebook's statistics endpoint uses a GET request, where you retrieve a specific record by its URL address.

Mixing up the request type causes the API to return HTTP 200 with a completely empty response, with no data and no error message. The `utils.py` module handles this automatically by keeping GET and POST settings separate. You just need to know this is why the code uses `tracker.get()` in this notebook instead of `tracker.post()`.

---
## Key Terms

**Aggregate statistic:** A single number that summarises many individual data points.
The median sale price is one statistic derived from all individual sale prices in a
suburb during a period.

**Median:** The middle value when all values are sorted. Half the values are above it,
half below. Less affected by extreme values (very expensive or very cheap properties)
than an average.

**Quarter:** A three-month period. Q1 = January-March, Q2 = April-June, Q3 = July-September,
Q4 = October-December.

**Time series:** Data collected at regular intervals over time. The output of this
endpoint is a time series: one row per quarter, going back up to 45 quarters.

**NaN (Not a Number):** The value Python uses when a number is missing or unknown.
In this context, a NaN in the `medianSoldPrice` column means there were not enough
sales in that quarter for the API to calculate a reliable median. This is normal,
not an error.

**DataFrame:** A table of data in Python. Rows are time periods (quarters), columns
are statistics. It behaves like an Excel spreadsheet and can be filtered, sorted,
and charted without making further API calls.

---
## How This Endpoint Is Different From the Listings Endpoint

| | Listings endpoint | Statistics endpoint |
|---|---|---|
| **What it returns** | Individual property records | Summary statistics per suburb |
| **Request type** | POST (submit a search form) | GET (retrieve by address) |
| **Records per call** | Up to 1,000 (may need cursor) | Exactly one row per time window |
| **Historical depth** | Back to ~2018 reliably | Up to 45 quarters (~11 years) |
| **Cost** | 1 credit per page | 1 credit per suburb-category pair |
| **Best for** | Individual property analysis | Market trend analysis |

These are two completely separate tools for two different research questions. Many
studies will use both.

---
## Setup: Check Packages

In [ ]:
missing = []
for pkg in ['requests', 'pandas', 'matplotlib', 'dotenv']:
    try:
        __import__(pkg)
    except ImportError:
        missing.append(pkg if pkg != 'dotenv' else 'python-dotenv')

if missing:
    print('Missing packages -- run in terminal:')
    print(f'  pip install {" ".join(missing)}')
else:
    print('All packages found. Setup OK.')

## Setup: Connect to the API

In [ ]:
import sys
from urllib.parse import quote  # used to make suburb names safe to put in a URL
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import os

if '.' not in sys.path:
    sys.path.insert(0, '.')

from utils import PROXY_BASE, APICallTracker

tracker = APICallTracker()

# The statistics endpoint has a different URL path from the listings endpoint
STATS_BASE = f'{PROXY_BASE}/v2/suburbPerformanceStatistics'


print(f'Connected to: {PROXY_BASE}')
print(f'Account:      {os.getenv("AURIN_USERNAME", "(not found -- check your .env file)")}')
print(f'Statistics endpoint: {STATS_BASE}')

---
## The Statistics Endpoint: What You Can Request

The URL for the statistics endpoint includes the suburb information directly in the address:

```
GET /v2/suburbPerformanceStatistics/{state}/{suburb}/{postcode}
```

For Carlton in Victoria with postcode 3053, that becomes:

```
GET /v2/suburbPerformanceStatistics/VIC/Carlton/3053
```

You also pass several query parameters (options):

| Parameter | What to put | Notes |
|---|---|---|
| `propertyCategory` | `House` or `Unit` | Required. Each category needs a separate call. |
| `periodSize` | `quarters`, `halfYears`, or `years` | Use `quarters` for most market analysis. |
| `totalPeriods` | 1 to 45 | How many quarters to return. Cost is always 1 credit (always use 45). |
| `startingPeriodRelativeToCurrent` | 0 = most recent quarter | Leave at 0 unless you have a reason to offset. |

> **`propertyCategory` is limited to `House` or `Unit` on this endpoint.** This is a hard constraint of the statistics API — unlike the listings endpoint in Notebook 1, which returns many property types (`ApartmentUnitFlat`, `Townhouse`, `Villa`, etc.), the statistics endpoint only aggregates at the House or Unit level. If you need statistics for a specific property type like townhouses, you would need to fetch individual listings from Notebook 1 and calculate your own aggregates.

**Key insight:** fetching `totalPeriods=45` (11 years of data) costs exactly the same as fetching `totalPeriods=1` (one quarter). There is no reason ever to request fewer than 45 periods.

---
## A Hidden Danger: The Silent Empty Response

If you misspell a suburb name or put in a wrong postcode, the API responds with HTTP 200 (the standard "success" signal) but sends back a completely empty body, with no data and no error message. Your code will not crash. It will just silently collect nothing for that suburb, and the credit is still spent.

The cell below demonstrates this deliberately using a misspelled suburb name. Watch the output carefully: the status code says 200 (success) but the body is empty.

In [ ]:
# Intentional typo: 'Cralton' instead of 'Carlton'
# This demonstrates what happens with a wrong suburb name
bad_suburb = 'Cralton'  # does not exist
bad_postcode = '3053'

# Build the URL (quote() makes the suburb name safe for a URL)
bad_url = f'{STATS_BASE}/VIC/{quote(bad_suburb, safe="")}/{bad_postcode}'

bad_params = {
    'propertyCategory': 'House',
    'periodSize': 'quarters',
    'totalPeriods': 1,  # keep low for this demo
    'startingPeriodRelativeToCurrent': 0,
}

# Make the request (uses GET, not POST)
r_bad = tracker.get(bad_url, params=bad_params)
tracker.checkpoint('Silent empty response demo')

print(f'Intentional typo: {bad_suburb} instead of Carlton')
print()
print(f'HTTP status code: {r_bad.status_code}  (200 means "success" -- misleadingly)')
print(f'Body length: {len(r_bad.text)} characters')
print(f'Body content: "{r_bad.text.strip()}"')
print()
print('The server said success, but sent nothing back.')
print('The credit was still consumed.')

### The guard check: detecting an empty response

Every function that calls the statistics endpoint should include this two-line check
immediately after receiving the response:

```python
if r.status_code == 200 and not r.text.strip():
    print(f'[{suburb}] Empty response -- check the suburb name and postcode.')
    return None  # stop and return nothing rather than crashing
```

**Line 1** confirms the server said "success" (status code 200).
**Line 2** checks whether any data was actually returned (`r.text.strip()` is an empty
string if the body is blank).

If both conditions are true, the response is a silent empty — print a warning and
skip that suburb rather than recording a blank result.

Returning `None` rather than raising an error is intentional and important. When you are fetching a list of suburbs in a loop, a single bad suburb name should not stop the entire run. The loop checks whether the return value is `None` and skips that suburb, while all successfully fetched suburbs continue to be collected and saved. This means only the one failed suburb needs to be re-run once you have corrected the spelling — not the whole list again, and without spending credits on suburbs you already have.

---
## Fetching Statistics for One Suburb

The `get_suburb_stats()` function below encapsulates all the steps: builds the URL,
sends the GET request, checks for the silent empty response, and returns the JSON data.

Run the cell to define the function. You will call it by name in the next step.

In [ ]:
VALID_STATES     = {'ACT', 'NSW', 'QLD', 'VIC', 'SA', 'WA', 'NT', 'TAS'}
VALID_CATEGORIES = {'House', 'Unit'}
VALID_PERIODS    = {'quarters', 'halfyears', 'years'}

def get_suburb_stats(suburb, state, postcode,
                     property_category='House',
                     period_size='quarters',
                     total_periods=45,
                     starting_period=0):
    """Fetch suburb performance statistics from the API.

    suburb:            suburb name (must match Domain's spelling exactly).
    state:             state code, e.g. 'VIC'. Case is ignored -- 'vic' and 'VIC' both work.
    postcode:          four-digit postcode as a string, e.g. '3053'.
    property_category: 'House' or 'Unit'. Case is ignored.
    period_size:       'quarters', 'halfYears', or 'years'. Case is ignored.
    total_periods:     integer between 1 and 45. Always use 45 to maximise data for the same credit cost.
    starting_period:   0 = most recent period; leave at 0 unless offsetting.

    Returns the API response as a Python dictionary, or None if validation fails,
    the request failed, or the response was empty.
    """
    # --- Input validation (all checks run before any API call is made) ---

    state             = state.upper()
    property_category = property_category.capitalize()
    period_size       = period_size.lower()

    if state not in VALID_STATES:
        print(f'  [{suburb}] Invalid state "{state}". Must be one of: {", ".join(sorted(VALID_STATES))}')
        return None

    if property_category not in VALID_CATEGORIES:
        print(f'  [{suburb}] Invalid property_category "{property_category}". Must be "House" or "Unit".')
        return None

    if period_size not in VALID_PERIODS:
        print(f'  [{suburb}] Invalid period_size "{period_size}". Must be "quarters", "halfYears", or "years".')
        return None

    if not isinstance(total_periods, int) or not (1 <= total_periods <= 45):
        print(f'  [{suburb}] Invalid total_periods "{total_periods}". Must be an integer between 1 and 45.')
        return None

    if not isinstance(starting_period, int) or starting_period < 0:
        print(f'  [{suburb}] Invalid starting_period "{starting_period}". Must be a non-negative integer.')
        return None

    # --- Build and send the request ---

    # quote() encodes multi-word suburb names (e.g. 'South Yarra') so they do not cause errors in the URL
    url = f'{STATS_BASE}/{state}/{quote(suburb, safe="")}/{postcode}'

    params = {
        'propertyCategory': property_category,
        'periodSize': period_size,
        'totalPeriods': total_periods,
        'startingPeriodRelativeToCurrent': starting_period,
    }

    # Use GET (not POST) for this endpoint
    r = tracker.get(url, params=params)

    # Guard: detect the silent empty response (wrong suburb name or postcode)
    if r.status_code == 200 and not r.text.strip():
        print(f'  [{suburb}] Empty response -- check suburb name and postcode.')
        return None

    if r.status_code != 200:
        print(f'  [{suburb}] HTTP {r.status_code}: {r.text[:120]}')
        return None

    return r.json()  # convert the JSON text into a Python dictionary


print('get_suburb_stats defined.')

### Convert the response to a table

The API response is a nested dictionary (data inside data inside data). The
`series_to_df()` function below unpacks it into a flat table (DataFrame) with
one row per quarter.

A DataFrame is like an Excel spreadsheet in Python: rows are time periods,
columns are statistics. Once data is in a DataFrame, all filtering, sorting,
and charting happens locally -- no more API calls needed.

In [ ]:
def series_to_df(api_response, suburb_name):
    """Convert a suburb statistics API response into a tidy table (DataFrame).

    api_response: the dictionary returned by get_suburb_stats().
    suburb_name:  a label to add as a column so multiple suburbs can be combined.

    Returns a DataFrame with one row per period. Fields the API did not report
    for a given period will appear as NaN (see the 'Handling Missing Values'
    section later in this notebook for what to do with these).
    """
    # The statistics live inside a nested structure: response -> series -> seriesInfo
    if not api_response or not api_response.get('series'):
        return pd.DataFrame()  # return an empty table if there is nothing to convert

    rows = []
    for entry in api_response['series']['seriesInfo']:
        row = {
            'suburb': suburb_name,
            'year':   entry['year'],   # the year of this period
            'month':  entry['month'],  # the starting month of this period
            # Create a proper date from year and month so charts have a proper time axis
            'date':   pd.Timestamp(year=entry['year'], month=entry['month'], day=1),
        }
        # The 'values' dict contains all the statistics for this period
        # (medianSoldPrice, numberSold, daysOnMarket, etc.)
        row.update(entry.get('values', {}))
        rows.append(row)

    # Sort oldest to newest so charts display correctly
    return pd.DataFrame(rows).sort_values('date').reset_index(drop=True)


print('series_to_df defined.')

### Fetch Carlton: 45 quarters in one call

The cell below fetches 45 quarters of house statistics for Carlton 3053.
**This costs 1 credit.**

No `property_category` is passed, so it defaults to `'House'`. To get Unit statistics for the same suburb, you would make a second call with `property_category='Unit'` — that costs another credit.

**Expected output:** a table with roughly 40-45 rows (the API returns however
many quarters of data it has, up to your requested maximum). Columns include
`suburb`, `year`, `month`, `date`, and several statistical fields.

In [ ]:
# Fetch 45 quarters of house statistics for Carlton (1 credit)
carlton_data = get_suburb_stats('Carlton', 'VIC', '3053', total_periods=45)
tracker.checkpoint('Carlton (45 quarters)')

if carlton_data:
    df_carlton = series_to_df(carlton_data, 'Carlton')
    print(f'Carlton: {len(df_carlton)} quarters returned')
    print(f'Earliest quarter: {df_carlton["date"].min().strftime("%Y-%m")}')
    print(f'Latest quarter:   {df_carlton["date"].max().strftime("%Y-%m")}')
    print()
    print('Columns in the dataset:')
    for col in df_carlton.columns:
        print(f'  {col}')
    print()
    df_carlton.head(5)  # show the first 5 rows

### Understanding the output table

The key statistical columns are:

- **`medianSoldPrice`:** the median sale price for all properties sold in that quarter.
- **`numberSold`:** how many properties sold.
- **`daysOnMarket`:** the median number of days between listing and sale.
- **`medianRentListingPrice`:** the median asking rent for rental listings.

Some of these columns may show `NaN` (missing) for some quarters. This means
there were not enough transactions in that quarter for the API to report a
reliable statistic. It is not an error. See the section on handling missing
values later in this notebook.

---
## Fetching Multiple Suburbs Efficiently

Fetching six suburbs with 45 quarters each costs **6 credits total**. If you had
instead fetched 1 quarter at a time and gone back 45 periods, the same data would
have cost 6 x 45 = 270 credits.

The cell below loops over a list of suburbs and concatenates all results into one
combined table. A message is printed after each suburb so you can follow progress.

In [ ]:
# List of suburbs to fetch: (suburb name, state, postcode)
SUBURBS = [
    ('Carlton',     'VIC', '3053'),
    ('Fitzroy',     'VIC', '3065'),
    ('Richmond',    'VIC', '3121'),
    ('Brunswick',   'VIC', '3056'),
    ('Northcote',   'VIC', '3070'),
    ('Collingwood', 'VIC', '3066'),
]

all_dfs = []  # collect one DataFrame per suburb

for suburb, state, postcode in SUBURBS:
    data = get_suburb_stats(suburb, state, postcode, total_periods=45)
    if data:
        df = series_to_df(data, suburb)
        all_dfs.append(df)
        print(f'  {suburb}: {len(df)} quarters fetched.')
    else:
        print(f'  {suburb}: no data returned (check suburb name and postcode).')

tracker.checkpoint('Six-suburb fetch (45 quarters each)')

# Combine all DataFrames into one big table
df_all = pd.concat(all_dfs, ignore_index=True)

print(f'\nCombined dataset: {len(df_all):,} rows ({len(SUBURBS)} suburbs x ~45 quarters)')
print(f'API calls made for this section: {len(SUBURBS)}')
print(f'Equivalent cost if fetched one quarter at a time: {len(SUBURBS) * 45} credits')
print(f'Actual cost:                                       {len(SUBURBS)} credits')
print(f'Total API calls so far: {tracker.total}')

---

## Save Your Results

Before doing any analysis, save the combined dataset to a CSV file.

Data fetched from the API only exists in memory while the notebook is running. If the notebook crashes, the kernel restarts, or your computer shuts down, that data is gone and you will need to call the API again — spending credits you have already used. Saving immediately after every fetch protects your work.

Run the cell below to save.

In [ ]:
# If you save somewhere other than this folder, give the full path, using
# forward slashes (/) or doubled backslashes (\\). A pasted Windows path with
# single backslashes may not work.
output_filename = 'inner_melbourne_suburb_statistics.csv'
df_all.to_csv(output_filename, index=False)

print(f'Saved: {output_filename}')
print(f'Rows:    {len(df_all):,}')
print(f'Columns: {list(df_all.columns)}')

---
## Creating Charts From the Data

All the analysis below uses the data already downloaded. No more API calls are
made. This is why fetching maximum historical depth upfront is always the right
strategy: you spend the same credits, but you get far more analytical flexibility.

### Chart 1: Median Sale Price Over Time

The next cell draws a line for each suburb showing how the median sale price
changed quarter by quarter. Quarters with no reported price (NaN) are
automatically skipped, so the line may have small gaps in low-volume periods.

In [ ]:
price_col = 'medianSoldPrice'

if price_col in df_all.columns:
    # Drop rows where price is missing -- cannot plot a missing value
    df_prices = df_all.dropna(subset=[price_col])

    fig, ax = plt.subplots(figsize=(12, 5))

    # Draw one line per suburb
    for suburb, group in df_prices.groupby('suburb'):
        ax.plot(
            group['date'],
            group[price_col] / 1_000_000,  # convert to millions for a readable y-axis
            label=suburb,
            marker='o',
            markersize=3,
        )

    ax.set_title('Median House Sale Price by Quarter, Inner Melbourne', fontsize=13)
    ax.set_xlabel('Quarter')
    ax.set_ylabel('Median sale price (AUD millions)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))  # show year on x-axis
    ax.xaxis.set_major_locator(mdates.YearLocator())
    plt.xticks(rotation=45)
    ax.legend(loc='upper left', fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print(f'Column "{price_col}" not found in the data.')
    print(f'Available columns: {list(df_all.columns)}')

### Chart 2: Quarterly Transaction Volume

The number of properties sold per quarter reveals seasonal patterns and
how market activity has changed over time across different suburbs.

In [ ]:
vol_col = 'numberSold'

if vol_col in df_all.columns:
    df_vol = df_all.dropna(subset=[vol_col])

    fig, ax = plt.subplots(figsize=(12, 5))

    for suburb, group in df_vol.groupby('suburb'):
        ax.plot(
            group['date'],
            group[vol_col],
            label=suburb,
            marker='o',
            markersize=3,
        )

    ax.set_title('Quarterly Transactions by Suburb, Inner Melbourne', fontsize=13)
    ax.set_xlabel('Quarter')
    ax.set_ylabel('Number of properties sold')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator())
    plt.xticks(rotation=45)
    ax.legend(loc='upper right', fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print(f'Column "{vol_col}" not found.')
    print(f'Available columns: {list(df_all.columns)}')

---
## Handling Missing Values (NaN Periods)

Some rows in the data have `NaN` (Not a Number) for certain columns. This means
there were not enough transactions in that quarter for the API to calculate a
reliable statistic. This is especially common for:

- Smaller suburbs with few transactions
- Quieter property categories (e.g. Units in a suburb that mainly has houses)
- Quarters at the beginning of the historical record when coverage was thinner

**NaN is not a bug.** Treat it as "not enough data to report," not as zero.

The cell below measures how often each key column has data (completeness) and
shows how to create a clean version of the dataset by removing incomplete rows.

In [ ]:
# Identify the key statistical columns (exclude the label and date columns)
stat_cols = [
    c for c in df_all.columns
    if c not in ('suburb', 'year', 'month', 'date')
]

# Measure completeness: what percentage of quarters have a value for each column?
print('Completeness by suburb (% of quarters with a reported value):')
print()

for suburb, group in df_all.groupby('suburb'):
    # Calculate the percentage of non-null values for the most important columns
    key_cols = [c for c in ['medianSoldPrice', 'numberSold', 'daysOnMarket'] if c in group.columns]
    if key_cols:
        completeness = (group[key_cols].notna().mean() * 100).round(1)
        row_str = '  '.join(f'{col}: {pct}%' for col, pct in completeness.items())
        print(f'  {suburb:<14} {row_str}')

In [ ]:
# Create a clean version of the dataset by removing quarters with missing key values
# This is appropriate for trend analysis where you need continuous data
key_cols = [c for c in ['medianSoldPrice', 'numberSold'] if c in df_all.columns]

if key_cols:
    # dropna removes rows where any of the listed columns has a missing value
    df_clean = df_all.dropna(subset=key_cols)

    print(f'Full dataset rows:  {len(df_all):,}')
    print(f'Clean dataset rows: {len(df_clean):,}  (dropped {len(df_all) - len(df_clean):,} rows with missing values)')
    print()
    print('Use df_clean for trend analysis.')
    print('Keep df_all if you need to analyse data availability patterns.')

---
## What If Something Goes Wrong?

**Empty response for a suburb ("check suburb name and postcode"):**
The suburb name must match Domain's spelling exactly, including capitalisation
and spaces. Check the suburb name on the Domain website by searching for a
property in that suburb and noting how Domain spells it.

**All columns are NaN for a suburb:**
The suburb and property category combination may have too few transactions
for any quarter. Try changing `property_category` from 'House' to 'Unit'
or vice versa. Some suburbs are predominantly one type.

**Chart shows no lines or a blank chart:**
This usually means all values in the column are NaN. Check the completeness
output above. If every suburb shows 0% completeness for a column, that column
may not be available for your combination of suburb and property category.

**Credentials error (account not found):**
Check your `.env` file in the main `domain_api` folder. Confirm
`AURIN_USERNAME` and `AURIN_PASSWORD` are correct.

---

## What's Next: Notebook 3

You can now pull suburb performance statistics for multiple suburbs efficiently, with years of quarterly data in a handful of credits, ready to chart and analyse without making another API call.

**Notebook 3** goes back to individual listings, but covers two advanced retrieval techniques. The first is cursor advancement, for areas with more than 1,000 listings where simple pagination silently truncates results. The second is checkpoint recovery, for large multi-suburb runs that need to survive interruptions without losing progress or re-spending credits.

→ Open `notebook-3-pagination-tricks.ipynb` to continue.

---
## Credit Summary

In [ ]:
tracker.summary()